In [2]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# load data
df = pd.read_csv("C:\\Users\\Dan\\Downloads\\speeches_sentiment_5000.csv").copy()

# sort by time to avoid leakage
df["speech_begin"] = pd.to_datetime(df["speech_begin"], errors="coerce")
df = df.sort_values("speech_begin").copy()

# train / val / test split
train_end = int(len(df) * 0.70)
val_end = int(len(df) * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

# features
num_cols = [
    "speech_duration_seconds",
    "hour_of_day",
    "day_of_week",
    "month",
    "hour_sin",
    "hour_cos",
    "sentiment_score_neg",
    "sentiment_score_neu",
    "sentiment_score_pos"
]

cat_cols = [
    "speaker_party",
    "activiteit_soort",
    "time_bin",
    "sentiment_label"
]

X_train = train_df[num_cols + cat_cols]
y_train = train_df["motion_passed"]

X_test = test_df[num_cols + cat_cols]
y_test = test_df["motion_passed"]

# preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            num_cols
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            cat_cols
        )
    ]
)

# model
logreg_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

# train
logreg_model.fit(X_train, y_train)

# predict
y_pred = logreg_model.predict(X_test)
y_proba = logreg_model.predict_proba(X_test)[:, 1]

# evaluation
print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.45      0.27      0.33       298
           1       0.62      0.78      0.69       452

    accuracy                           0.58       750
   macro avg       0.53      0.52      0.51       750
weighted avg       0.55      0.58      0.55       750

ROC-AUC: 0.5260661044129001
Confusion matrix:
 [[ 79 219]
 [ 98 354]]
